# 04 — Data-quality validation and the Data Quality Reviewer agent

**Objectives**
1. Run the **deterministic data-quality engine** (50 rules) on the clean and corrupted datasets and produce auditable JSON + Markdown reports.
2. Measure detection against the defect catalog: every planted defect found, no false alarms.
3. See the difference between **fail**, **warning** and **error** — an error is never a pass.
4. Run the **Data Quality Reviewer agent** (Gemini, structured output) that explains the report, and see the **guardrails** that reject unfaithful reviews.
5. Exercise retry, revision, escalation and error handling offline, then live with your Gemini key.

**Prerequisites:** notebooks 00–02 done; `python scripts/generate_data.py` has been run. Python **3.11+** (required by `google-genai` 2.x). Dependencies installed from the updated `requirements.txt`.

**Configuration:** none for Parts 1–3. Part 4 (live Gemini) needs `GEMINI_API_KEY` in `.env` — without it those cells skip cleanly. BigQuery is **not** used yet: the engine runs locally on the same Parquet files you will upload in Stage 3; the BigQuery SQL backend with the same rule IDs comes in Stage 5.

## How the pieces fit

```mermaid
flowchart LR
    F[(Parquet files<br/>clean / corrupted)] --> E[Deterministic engine<br/>50 rules, stable IDs]
    E --> R[ValidationReport<br/>pass / fail / warning / error]
    R --> P[Minimal, redacted payload<br/>non-passing rules only]
    P --> G[Gemini reviewer<br/>structured JSON]
    G --> C{Guardrails<br/>deterministic checks}
    C -->|pass| A[Accepted review]
    C -->|fail, once| G
    C -->|fail again / API error| D[Deterministic review<br/>+ escalate to human]
    R -.->|statuses can never be changed by the LLM| A
```

The engine decides **what is wrong**. The reviewer explains **what it means and what to do**. Guardrails make sure the explanation is faithful to the engine.

## Part 0 — Setup

In [1]:
# Locate the project root (folder containing pyproject.toml) so the notebook works
# no matter which folder VS Code starts the kernel in, then make `src/` importable.
import os, sys
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    if root.parent == root:
        raise RuntimeError("Could not find pyproject.toml - open the analytics-agent folder in VS Code.")
    root = root.parent
os.chdir(root)
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
print("Project root:", root)
print("Python:", sys.version.split()[0], "| executable:", sys.executable)

Project root: /home/claude/analytics-agent
Python: 3.12.3 | executable: /home/claude/adk-venv/bin/python


In [2]:
import json, hashlib
import pandas as pd
from pathlib import Path
from analytics_agent.config import load_settings
from analytics_agent.data_generation.generator import GenerationConfig
from analytics_agent.data_quality import (RULE_SPECS, validate_tables, load_tables_from_files, detection_metrics)
from analytics_agent.agents.data_quality_reviewer import (
    DataQualityReviewer, DataQualityReview, deterministic_review, check_review,
    build_review_payload, build_prompt, review_to_markdown, SYSTEM_INSTRUCTION)
from analytics_agent.llm import GeminiStructuredClient, LLMError
from analytics_agent.testing import ScriptedGeminiClient, fake_api_error
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_columns", 20)

settings = load_settings(require_project=False)      # BigQuery project not needed in this notebook
AS_OF = GenerationConfig().as_of_ts
print("Gemini model :", settings.gemini.model)
print("Gemini key   :", "set" if settings.gemini_key_present else "not set (live cells will be skipped)")
print("Snapshot     :", AS_OF)

Gemini model : gemini-3.8-flash
Gemini key   : not set (live cells will be skipped)
Snapshot     : 2026-07-03 00:00:00+00:00


Checkpoint: the key is reported only as *set / not set* — its value is never printed.

## Part 1 — Deterministic validation

### Step 1.1 — The rule catalogue

50 rules: 7 schema checks, 41 integrity/consistency rules (same IDs as the Stage 2 oracle) and 2 soft warning rules (freshness, volume). Each has a stable ID, severity, expected condition and a recommended action.

In [3]:
rules = pd.DataFrame([{"rule_id": s.rule_id, "table": s.table, "severity": s.severity, "kind": s.kind,
                       "expected_condition": s.expected_condition} for s in RULE_SPECS])
print(rules.kind.value_counts().to_dict(), "| by severity:", rules.severity.value_counts().to_dict())
rules.head(12)

{'fail_rule': 48, 'warning_rule': 2} | by severity: {'high': 19, 'critical': 17, 'medium': 13, 'low': 1}


,rule_id,table,severity,kind,expected_condition
0,SCHEMA_COLUMNS_CUSTOMERS,customers,critical,fail_rule,no missing or unexpected columns
1,SCHEMA_COLUMNS_PRODUCTS,products,critical,fail_rule,no missing or unexpected columns
2,SCHEMA_COLUMNS_ORDERS,orders,critical,fail_rule,no missing or unexpected columns
3,SCHEMA_COLUMNS_ORDER_ITEMS,order_items,critical,fail_rule,no missing or unexpected columns
4,SCHEMA_COLUMNS_PAYMENTS,payments,critical,fail_rule,no missing or unexpected columns
5,SCHEMA_COLUMNS_SHIPMENTS,shipments,critical,fail_rule,no missing or unexpected columns
6,SCHEMA_COLUMNS_REFUNDS,refunds,critical,fail_rule,no missing or unexpected columns
7,CUS_PK_UNIQUE,customers,critical,fail_rule,0 violating records (customer_id)
8,CUS_REQUIRED_NOT_NULL,customers,high,fail_rule,0 violating records (customer_id)
9,CUS_REGION_ALLOWED,customers,medium,fail_rule,0 violating records (customer_id)


### Step 1.2 — Load the datasets from disk

These are the exact files that will be uploaded to BigQuery.

In [4]:
clean = load_tables_from_files("clean", "parquet")
corrupted = load_tables_from_files("corrupted", "parquet")
pd.DataFrame({"clean": {k: len(v) for k, v in clean.items()}, "corrupted": {k: len(v) for k, v in corrupted.items()}})

,clean,corrupted
customers,2000,2005
products,250,250
orders,15000,15000
order_items,32823,32840
payments,15804,15819
shipments,14291,14291
refunds,2047,2047


### Step 1.3 — Validate the clean dataset

**Checkpoint:** 50 pass, 0 fail, 0 warning, 0 error.

In [5]:
rep_clean = validate_tables(clean, AS_OF, dataset_label="clean")
print(rep_clean.run_id, rep_clean.counts(), "overall:", rep_clean.overall_status)
assert rep_clean.counts() == {"pass": 50, "fail": 0, "warning": 0, "error": 0}

dq_20261030T050236Z_baec16 {'pass': 50, 'fail': 0, 'warning': 0, 'error': 0} overall: pass


### Step 1.4 — Validate the corrupted dataset

**Checkpoint:** 20 rules fail (the 15 defects plus their documented cascades).

In [6]:
rep_bad = validate_tables(corrupted, AS_OF, dataset_label="corrupted")
print(rep_bad.run_id, rep_bad.counts(), "overall:", rep_bad.overall_status)
cols = ["rule_id", "table", "severity", "status", "observed_count", "rows_checked", "example_ids"]
rep_bad.summary_frame().query("status != 'pass'")[cols].sort_values(["severity", "rule_id"])

dq_20261030T050236Z_604796 {'pass': 30, 'fail': 20, 'warning': 0, 'error': 0} overall: fail


,rule_id,table,severity,status,observed_count,rows_checked,example_ids
7,CUS_PK_UNIQUE,customers,critical,fail,5,2005,"[CUS000192, CUS000705, CUS000766, CUS001219, CUS001813]"
23,ITM_PK_UNIQUE,order_items,critical,fail,12,32840,"[ITM00002344, ITM00009253, ITM00009631, ITM00014255, ITM00023525]"
34,PAY_CAPTURED_MATCHES_ORDER,payments,critical,fail,15,15819,"[ORD0000255, ORD0000727, ORD0000850, ORD0000944, ORD0003413]"
30,PAY_TXN_UNIQUE,payments,critical,fail,15,15819,"[txn_162944dbbf4b7215, txn_1e30cd6f9b4e17d8, txn_1ed1a475dc12fe25, txn_1f1155385421993..."
46,REF_WITHIN_REFUNDABLE,refunds,critical,fail,6,2047,"[REF000186, REF000199, REF000208, REF000228, REF000587]"
8,CUS_REQUIRED_NOT_NULL,customers,high,fail,10,2005,"[CUS000102, CUS000391, CUS000626, CUS000672, CUS000692]"
28,ITM_AMOUNTS_CONSISTENT,order_items,high,fail,5,32840,"[ITM00003673, ITM00005733, ITM00007483, ITM00015213, ITM00022701]"
24,ITM_LINE_UNIQUE,order_items,high,fail,12,32840,"[ITM00002344, ITM00009253, ITM00009631, ITM00014255, ITM00023525]"
25,ITM_ORDER_FK,order_items,high,fail,5,32840,"[ITM90000000, ITM90000001, ITM90000002, ITM90000003, ITM90000004]"
27,ITM_QUANTITY_RANGE,order_items,high,fail,5,32840,"[ITM00003673, ITM00005733, ITM00007483, ITM00015213, ITM00022701]"


Each result carries the expected condition, examples and a recommended next action:

In [7]:
r = rep_bad.get("PAY_TXN_UNIQUE")
print(json.dumps({k: getattr(r, k) for k in ["rule_id", "status", "severity", "observed_count", "expected_condition",
                                             "example_ids", "recommended_action", "business_impact"]}, indent=1))

{
 "rule_id": "PAY_TXN_UNIQUE",
 "status": "fail",
 "severity": "critical",
 "observed_count": 15,
 "expected_condition": "0 violating records (provider_transaction_id)",
 "example_ids": [
  "txn_162944dbbf4b7215",
  "txn_1e30cd6f9b4e17d8",
  "txn_1ed1a475dc12fe25",
  "txn_1f11553854219935",
  "txn_2949146d05b74e7b"
 ],
 "recommended_action": "Deduplicate payments by provider_transaction_id and reconcile with the payment provider's settlement report.",
 "business_impact": "Total payments received is overstated by the duplicated amounts."
}


### Step 1.5 — Detection quality against the defect catalog

Record-level comparison: for every rule, are the flagged records exactly the records the catalog says were corrupted? **Checkpoint:** missed = 0, false positives = 0 on corrupted; 0 non-passing rules on clean.

In [8]:
catalog = json.loads(Path("data/manifests/defect_catalog.json").read_text())
m = detection_metrics(rep_bad, catalog)
tp, exp, fp = m.true_pos.sum(), m.expected.sum(), m.false_pos.sum()
print(f"Defective records found: {tp}/{exp}  (recall {tp/exp:.0%})   false positives: {fp}   "
      f"precision {tp/(tp+fp):.0%}")
print("Non-passing rules on CLEAN data (false alarms):", len(rep_clean.by_status("fail", "warning", "error")))
assert m.missed.sum() == 0 and fp == 0
m

Defective records found: 154/154  (recall 100%)   false positives: 0   precision 100%
Non-passing rules on CLEAN data (false alarms): 0


,rule_id,status,expected,found,true_pos,missed,false_pos
0,CUS_PK_UNIQUE,fail,5,5,5,0,0
1,CUS_REQUIRED_NOT_NULL,fail,10,10,10,0,0
2,CUS_EMAIL_UNIQUE,fail,5,5,5,0,0
3,PRD_SKU_UNIQUE,fail,2,2,2,0,0
4,PRD_CATEGORY_ALLOWED,fail,3,3,3,0,0
5,ORD_STATUS_ALLOWED,fail,6,6,6,0,0
6,ORD_CUSTOMER_FK,fail,8,8,8,0,0
7,ORD_TOTAL_CONSISTENT,fail,10,10,10,0,0
8,ORD_SUBTOTAL_MATCHES_ITEMS,fail,12,12,12,0,0
9,ITM_PK_UNIQUE,fail,12,12,12,0,0


Defect-level view — which rule caught each catalogued defect:

In [9]:
pd.DataFrame([{"defect": d["defect_id"], "type": d["defect_type"], "detection_rule": d["detection_rule"],
               "caught": rep_bad.get(d["detection_rule"]).status == "fail",
               "records": rep_bad.get(d["detection_rule"]).observed_count} for d in catalog])

,defect,type,detection_rule,caught,records
0,D01,duplicate_primary_key,CUS_PK_UNIQUE,True,5
1,D02,missing_required_value,CUS_REQUIRED_NOT_NULL,True,10
2,D03,invalid_category,PRD_CATEGORY_ALLOWED,True,3
3,D04,duplicate_business_key,PRD_SKU_UNIQUE,True,2
4,D05,orphan_foreign_key,ORD_CUSTOMER_FK,True,8
5,D06,invalid_status,ORD_STATUS_ALLOWED,True,6
6,D07,incorrect_monetary_total,ORD_TOTAL_CONSISTENT,True,10
7,D08,negative_quantity,ITM_QUANTITY_RANGE,True,5
8,D09,orphan_foreign_key,ITM_ORDER_FK,True,5
9,D10,duplicate_rows,ITM_PK_UNIQUE,True,12


### Step 1.6 — Warning vs fail vs error

* **warning** — a soft threshold is crossed (suspicious, not proven wrong). Simulated by validating with a snapshot 30 days later: the data now looks stale.
* **error** — a rule could not run. Simulated by dropping a column. The engine reports `error`, never a silent pass.

In [10]:
rep_stale = validate_tables(clean, AS_OF + pd.Timedelta(days=30), dataset_label="clean (stale snapshot)")
print("stale :", rep_stale.counts(), "->", rep_stale.overall_status, "|", rep_stale.get("FRESH_ORDERS").example_ids)

broken = dict(clean); broken["shipments"] = clean["shipments"].drop(columns=["delivered_timestamp"])
rep_broken = validate_tables(broken, AS_OF, dataset_label="clean (column dropped)")
print("broken:", rep_broken.counts(), "->", rep_broken.overall_status)
rep_broken.summary_frame().query("status != 'pass'")[["rule_id", "status", "observed_count", "example_ids", "error_message"]]

stale : {'pass': 49, 'fail': 0, 'warning': 1, 'error': 0} -> warning | ['latest_order=2026-06-30;lag_days=32']


broken: {'pass': 47, 'fail': 1, 'warning': 0, 'error': 2} -> error


,rule_id,status,observed_count,example_ids,error_message
5,SCHEMA_COLUMNS_SHIPMENTS,fail,1,[missing:delivered_timestamp],
39,SHP_DELIVERED_TS_CONSISTENT,error,0,[],AttributeError: 'DataFrame' object has no attribute 'delivered_timestamp'
40,SHP_DELIVERY_AFTER_SHIP,error,0,[],AttributeError: 'DataFrame' object has no attribute 'delivered_timestamp'


### Step 1.7 — Save the reports

JSON is machine-readable (for agents, dashboards, tests); Markdown is for people. Saved under `reports/dq/` (git-ignored).

In [11]:
for rep in (rep_clean, rep_bad):
    j, mdp = rep.save("reports/dq")
    print("saved", j, "and", mdp.name)
print()
print(rep_bad.to_markdown()[:1500], "...")

saved reports/dq/dq_20261030T050236Z_baec16.json and dq_20261030T050236Z_baec16.md
saved reports/dq/dq_20261030T050236Z_604796.json and dq_20261030T050236Z_604796.md

# Data-quality report — corrupted

Run `dq_20261030T050236Z_604796` at 2026-10-30T05:02:36+00:00 · source: local · snapshot as_of 2026-07-03 00:00:00+00:00

**Overall: FAIL** — 30 pass, 20 fail, 0 warning, 0 error

| Rule | Table | Severity | Status | Observed | Expected | Examples | Action |
|---|---|---|---|---|---|---|---|
| `CUS_PK_UNIQUE` | customers | critical | fail | 5 | 0 violating records (customer_id) | CUS000192, CUS000705, CUS000766 | Deduplicate customers in the source CRM sync; keep the latest record per customer_id and add a uniqueness check to the load. |
| `ITM_PK_UNIQUE` | order_items | critical | fail | 12 | 0 violating records (order_item_id) | ITM00002344, ITM00009253, ITM00009631 | Remove duplicate order lines (exact re-loads) and make the load idempotent. |
| `PAY_CAPTURED_MATCHES_ORDER` | payments

## Part 2 — What the reviewer agent receives

Only **non-passing rules**, at most 3 example IDs each, **no raw rows**, and emails replaced by a stable hash. Data sent to Gemini leaves your machine and is processed by Google under the Gemini API terms, so the payload is kept minimal.

In [12]:
payload = build_review_payload(rep_bad)
text = json.dumps(payload)
print(f"rules sent: {len(payload['non_passing_rules'])} of {len(rep_bad.results)} | payload size: {len(text):,} chars")
print("contains '@' (an email)?", "@" in text)
assert "@" not in text
print(json.dumps(payload["non_passing_rules"][:2], indent=1))

rules sent: 20 of 50 | payload size: 12,098 chars
contains '@' (an email)? False
[
 {
  "rule_id": "CUS_PK_UNIQUE",
  "table": "customers",
  "severity": "critical",
  "status": "fail",
  "description": "customer_id is unique",
  "expected_condition": "0 violating records (customer_id)",
  "observed_count": 5,
  "rows_checked": 2005,
  "example_ids": [
   "CUS000192",
   "CUS000705",
   "CUS000766"
  ],
  "error_message": "",
  "known_business_impact": "Joins from orders to customers duplicate those customers' orders, inflating regional/segment sales and AOV.",
  "affected_metrics_hint": [
   "net sales by region",
   "net sales by segment",
   "AOV by region"
  ],
  "suggested_action": "Deduplicate customers in the source CRM sync; keep the latest record per customer_id and add a uniqueness check to the load."
 },
 {
  "rule_id": "CUS_REQUIRED_NOT_NULL",
  "table": "customers",
  "severity": "high",
  "status": "fail",
  "description": "Required customer columns are not null",
  "expe

The system instruction (fixed rules for the model). The report is wrapped in `<dq_report>` tags and explicitly treated as data, so text inside it cannot act as instructions:

In [13]:
print(SYSTEM_INSTRUCTION)

You are a data-quality reviewer for an e-commerce analytics warehouse.
You receive the output of a deterministic validation engine inside <dq_report> tags.
That engine is the source of truth.

Rules you must follow:
1. Only use facts in the report. Never invent rules, counts, record identifiers, causes or tables.
2. Cite rule_id values exactly as written. Every rule with status fail, warning or error must be covered by a finding.
3. Never lower a severity below the rule's deterministic severity. You may group rules; a group's severity is the highest of its rules.
4. classification is confirmed_failure if any cited rule has status fail, otherwise suspicious_pattern.
5. Each evidence item must quote a rule_id and its observed_count from the report.
6. overall_assessment: fit_for_use only if no rule failed or errored; cannot_assess if any rule has status error;
   not_fit_for_use if any critical rule failed; otherwise use_with_caveats.
7. Set requires_human_review true for any finding wit

## Part 3 — Reviewer behaviour offline (no API key, no cost)

### Step 3.1 — Deterministic baseline review

A template-based review with no LLM. It is the fallback when Gemini is unavailable and the baseline to compare against. It must itself pass the guardrails.

In [14]:
base = deterministic_review(rep_bad)
g = check_review(base, rep_bad)
print("overall:", base.overall_assessment, "| findings:", len(base.findings), "| guardrails passed:", g.passed)
pd.DataFrame([{"title": f.title, "severity": f.severity, "classification": f.classification,
               "rules": ", ".join(f.rule_ids), "human_review": f.requires_human_review} for f in base.findings])

overall: not_fit_for_use | findings: 7 | guardrails passed: True


,title,severity,classification,rules,human_review
0,customers: 3 rule(s) not passing,critical,confirmed_failure,"CUS_PK_UNIQUE, CUS_REQUIRED_NOT_NULL, CUS_EMAIL_UNIQUE",True
1,products: 2 rule(s) not passing,high,confirmed_failure,"PRD_SKU_UNIQUE, PRD_CATEGORY_ALLOWED",True
2,orders: 4 rule(s) not passing,high,confirmed_failure,"ORD_CUSTOMER_FK, ORD_SUBTOTAL_MATCHES_ITEMS, ORD_TOTAL_CONSISTENT, ORD_STATUS_ALLOWED",True
3,order_items: 5 rule(s) not passing,critical,confirmed_failure,"ITM_PK_UNIQUE, ITM_AMOUNTS_CONSISTENT, ITM_LINE_UNIQUE, ITM_ORDER_FK, ITM_QUANTITY_RANGE",True
4,payments: 3 rule(s) not passing,critical,confirmed_failure,"PAY_CAPTURED_MATCHES_ORDER, PAY_TXN_UNIQUE, PAY_AFTER_ORDER",True
5,shipments: 2 rule(s) not passing,high,confirmed_failure,"SHP_DELIVERED_TS_CONSISTENT, SHP_DELIVERY_AFTER_SHIP",True
6,refunds: 1 rule(s) not passing,critical,confirmed_failure,REF_WITHIN_REFUNDABLE,True


Clean data: nothing to review, so the reviewer **abstains** and makes no LLM call:

In [15]:
res_clean = DataQualityReviewer(llm=None).review(rep_clean)
print(res_clean.outcome, "|", res_clean.review.overall_assessment, "|", res_clean.review.summary)

abstained | fit_for_use | Deterministic review of dataset 'clean': 0 failing, 0 warning and 0 errored rules out of 50. No defects were detected by the rule set.


### Step 3.2 — Guardrails reject unfaithful reviews

Each case below takes a valid review and introduces one realistic LLM mistake. **Checkpoint:** every case is rejected with a specific reason.

In [16]:
def broken_review(fn):
    r = base.model_copy(deep=True); fn(r); return r

cases = {
    "hallucinated rule":        lambda r: r.findings[0].rule_ids.append("ORD_GHOST_RULE"),
    "dropped a failing rule":   lambda r: setattr(r, "findings", r.findings[1:]),
    "downgraded severity":      lambda r: setattr(next(f for f in r.findings if f.severity == "critical"), "severity", "low"),
    "declared bad data fit":    lambda r: setattr(r, "overall_assessment", "fit_for_use"),
    "cited a passing rule":     lambda r: r.findings[0].rule_ids.append("PRD_PK_UNIQUE"),
    "evidence without numbers": lambda r: setattr(r.findings[0], "evidence", ["Looks problematic."]),
    "no human review on critical": lambda r: setattr(r, "requires_human_review", False),
}
rows = []
for name, fn in cases.items():
    res = check_review(broken_review(fn), rep_bad)
    rows.append({"mistake": name, "rejected": not res.passed, "first reason": res.violations[0] if res.violations else ""})
    assert not res.passed, name
pd.DataFrame(rows)

,mistake,rejected,first reason
0,hallucinated rule,True,finding 1 ('customers: 3 rule(s) not passing') cites unknown rule 'ORD_GHOST_RULE' (no...
1,dropped a failing rule,True,"non-passing rules not covered by any finding: ['CUS_EMAIL_UNIQUE', 'CUS_PK_UNIQUE', 'C..."
2,downgraded severity,True,finding 1 ('customers: 3 rule(s) not passing') severity 'low' is lower than determinis...
3,declared bad data fit,True,overall_assessment 'fit_for_use' not allowed by the report; expected one of ['not_fit_...
4,cited a passing rule,True,"finding 1 ('customers: 3 rule(s) not passing') cites PRD_PK_UNIQUE, which PASSED; only..."
5,evidence without numbers,True,finding 1 ('customers: 3 rule(s) not passing') evidence does not quote any observed_co...
6,no human review on critical,True,critical failures present but overall requires_human_review is false


### Step 3.3 — The full review loop with a scripted model

`ScriptedGeminiClient` replays prepared responses instead of calling Gemini (clearly a test double, not a model). It lets you watch the control flow:

* **accepted** — first answer passes guardrails;
* **revised** — first answer rejected, the violations are sent back, the second passes;
* **escalated** — still invalid after the bounded revision → deterministic review + human review;
* transient API errors are retried (bounded); quota and auth errors fail fast.

In [17]:
good = base.model_dump_json()
bad = broken_review(cases["declared bad data fit"]).model_dump_json()
def run(script, **kw):
    fake = ScriptedGeminiClient(script)
    llm = GeminiStructuredClient("scripted-model", client=fake, sleep=lambda s: None, **kw)
    return DataQualityReviewer(llm, max_revisions=1).review(rep_bad), fake

scenarios = {
    "good answer":                       [good],
    "bad, then good":                    [bad, good],
    "bad twice":                         [bad, bad],
    "malformed JSON, then good":         ["{oops", good],
    "rate limit, then good":             [fake_api_error(429, "RESOURCE_EXHAUSTED", "Rate limit exceeded"), good],
    "quota exhausted":                   [fake_api_error(429, "RESOURCE_EXHAUSTED", "You exceeded your current quota")],
    "invalid API key":                   [fake_api_error(400, "INVALID_ARGUMENT", "API key not valid")],
    "unknown model":                     [fake_api_error(404, "NOT_FOUND", "models/x is not found")],
    "server down x3":                    [fake_api_error(503, "UNAVAILABLE", "overloaded")] * 3,
}
rows = []
for name, script in scenarios.items():
    res, fake = run(script)
    rows.append({"scenario": name, "outcome": res.outcome, "reviewer": res.reviewer, "model calls": len(fake.calls),
                 "human review": res.requires_human_review, "error": (res.error or "")[:60]})
pd.DataFrame(rows)

,scenario,outcome,reviewer,model calls,human review,error
0,good answer,accepted,gemini:scripted-model,1,True,
1,"bad, then good",revised,gemini:scripted-model,2,True,
2,bad twice,escalated,deterministic,2,True,LLM review failed guardrails after bounded revisions
3,"malformed JSON, then good",revised,gemini:scripted-model,2,True,
4,"rate limit, then good",accepted,gemini:scripted-model,2,True,
5,quota exhausted,escalated,deterministic,1,True,"quota: Error code: 429 - {'error': {'code': 429, 'status': '"
6,invalid API key,escalated,deterministic,1,True,"auth: Error code: 400 - {'error': {'code': 400, 'status': 'I"
7,unknown model,escalated,deterministic,1,True,"model_not_found: Error code: 404 - {'error': {'code': 404, '"
8,server down x3,escalated,deterministic,3,True,"unavailable: Error code: 503 - {'error': {'code': 503, 'stat"


Checkpoints: *bad, then good* → `revised` with 2 calls; *bad twice* → `escalated` after exactly 2 calls (bounded); *rate limit* → retried; *quota*, *invalid key*, *unknown model* → 1 call then escalated (no pointless retries); *server down ×3* → 3 calls (1 + 2 retries) then escalated. In every escalation the user still gets the deterministic review — never a fabricated one.

In [18]:
res, fake = run([bad, good])
print("Second request includes the rejection feedback:")
print(fake.calls[1]["input"][-500:])

Second request includes the rejection feedback:
ential financial loss.",
   "affected_metrics_hint": [
    "net sales",
    "refund rate",
    "product refund rate"
   ],
   "suggested_action": "Freeze and audit these refunds with finance (possible error or fraud); enforce refund <= refundable amount."
  }
 ]
}
</dq_report>

Your previous review was REJECTED by automated checks for these reasons. Produce a corrected review that fixes all of them:
- overall_assessment 'fit_for_use' not allowed by the report; expected one of ['not_fit_for_use']


## Part 4 — Live review with Gemini

Runs only if `GEMINI_API_KEY` is set in `.env`. To get a key: open Google AI Studio → **Get API key** → create a key → put `GEMINI_API_KEY=...` in `.env` (never in the notebook). Restart the kernel after editing `.env`.

Cost: one review sends ~3–5k input tokens and returns a few thousand output tokens. The smoke test is a few dozen tokens.

In [19]:
RUN_LIVE = settings.gemini_key_present
if not RUN_LIVE:
    print("GEMINI_API_KEY not set - skipping Part 4. Add it to .env and restart the kernel to run live.")

GEMINI_API_KEY not set - skipping Part 4. Add it to .env and restart the kernel to run live.


### Step 4.1 — Smoke test (does the key, model and structured output work?)

In [20]:
from typing import Literal
from pydantic import BaseModel

class Ping(BaseModel):
    status: Literal["ok"]
    note: str

if RUN_LIVE:
    try:
        llm = GeminiStructuredClient(settings.gemini.model, timeout_s=settings.gemini.timeout_seconds,
                                     max_retries=settings.gemini.max_retries)
        ping, meta = llm.generate(system_instruction="Reply with the requested JSON only.",
                                  prompt='Return status "ok" and a five-word note.', schema=Ping, max_output_tokens=512)
        print("Smoke test OK:", ping, "| attempts", meta.attempts, "| latency", meta.latency_s, "s",
              "| tokens in/out", meta.input_tokens, meta.output_tokens)
    except LLMError as e:
        RUN_LIVE = False
        print(f"Smoke test failed: {e.kind} - {e.message}")
        print({"not_configured": "Set GEMINI_API_KEY in .env.",
               "auth": "Key invalid or restricted: create a new key in AI Studio.",
               "model_not_found": "Set GEMINI_MODEL in .env to a model listed at ai.google.dev/gemini-api/docs/models.",
               "quota": "Free-tier or billing quota exhausted: check AI Studio usage/billing.",
               "rate_limit": "Too many requests: wait a minute and re-run.",
               "network": "No connection to generativelanguage.googleapis.com (proxy/firewall/VPN?).",
               }.get(e.kind, "See the error message above."))

### Step 4.2 — Live review of the corrupted dataset

The answer goes through the same guardrails as in Part 3. Whatever the model returns, the outcome is one of accepted / revised / escalated.

In [21]:
if RUN_LIVE:
    before = json.dumps(rep_bad.to_dict()["results"])
    live = DataQualityReviewer(llm, max_revisions=1, max_output_tokens=settings.gemini.max_output_tokens).review(rep_bad)
    assert json.dumps(rep_bad.to_dict()["results"]) == before      # the reviewer cannot change statuses
    print("outcome:", live.outcome, "| reviewer:", live.reviewer, "| guardrails passed:", live.guardrail.passed,
          "| attempts:", live.attempts, "| human review:", live.requires_human_review)
    for i, v in enumerate(live.rejected_attempts, 1):
        print(f"rejected attempt {i}:", v)
    for mt in live.llm_meta:
        if mt: print("call:", mt.model, "attempts", mt.attempts, "latency", mt.latency_s, "s tokens in/out/thought",
                     mt.input_tokens, mt.output_tokens, mt.thought_tokens, "errors", mt.errors)
    print("\noverall:", live.review.overall_assessment, "\n", live.review.summary)

In [22]:
if RUN_LIVE:
    display(pd.DataFrame([{"title": f.title, "severity": f.severity, "classification": f.classification,
                           "rules": ", ".join(f.rule_ids), "affected metrics": ", ".join(f.affected_metrics)}
                          for f in live.review.findings]))

### Step 4.3 — Does the review cover every planted defect?

A defect counts as covered if its detection rule is cited by a finding. Guardrails already force full coverage of failing rules, so this should be 15/15 for any accepted review.

In [23]:
def coverage(review):
    cited = {rid for f in review.findings for rid in f.rule_ids}
    return sum(d["detection_rule"] in cited for d in catalog), len(catalog)

print("deterministic baseline covers %d/%d defects" % coverage(base))
if RUN_LIVE:
    print("live review covers %d/%d defects" % coverage(live.review))

deterministic baseline covers 15/15 defects


### Step 4.4 — Save the review

In [24]:
final = live if RUN_LIVE else DataQualityReviewer(llm=None).review(rep_bad)
out = Path("reports/dq") / f"{rep_bad.run_id}_review.md"
out.write_text(review_to_markdown(final))
(out.with_suffix(".json")).write_text(final.review.model_dump_json(indent=2))
print("saved", out, "| outcome:", final.outcome, "| reviewer:", final.reviewer)
print(review_to_markdown(final)[:1500])

saved reports/dq/dq_20261030T050236Z_604796_review.md | outcome: deterministic_only | reviewer: deterministic
# Data-quality review

Outcome: **deterministic_only** · reviewer: deterministic · guardrails: passed · human review required: True

**Overall assessment: not_fit_for_use**

Deterministic review of dataset 'corrupted': 20 failing, 0 warning and 0 errored rules out of 50. Critical failures: CUS_PK_UNIQUE, ITM_PK_UNIQUE, PAY_TXN_UNIQUE, PAY_CAPTURED_MATCHES_ORDER, REF_WITHIN_REFUNDABLE. Metrics listed under each finding should not be reported without caveats.

## 1. customers: 3 rule(s) not passing — critical (confirmed_failure)

Rules: `CUS_PK_UNIQUE`, `CUS_REQUIRED_NOT_NULL`, `CUS_EMAIL_UNIQUE`

Evidence:
- CUS_PK_UNIQUE: status fail, observed_count 5 of 2,005 rows (expected: 0 violating records (customer_id)); e.g. CUS000192, CUS000705, CUS000766
- CUS_REQUIRED_NOT_NULL: status fail, observed_count 10 of 2,005 rows (expected: 0 violating records (customer_id)); e.g. CUS000102,

### Step 4.5 — Optional: live error-handling checks

Set `RUN_LIVE_ERROR_TESTS = True` to confirm that a wrong key and an unknown model are classified correctly by the real API. These calls fail immediately and are not billed.

In [25]:
RUN_LIVE_ERROR_TESTS = False
if RUN_LIVE_ERROR_TESTS:
    for label, kw in [("invalid key", dict(api_key="invalid-key-for-testing")), ("unknown model", dict())]:
        model = "gemini-does-not-exist" if label == "unknown model" else settings.gemini.model
        try:
            GeminiStructuredClient(model, max_retries=0, **kw).generate(
                system_instruction="Reply with JSON.", prompt="ping", schema=Ping, max_output_tokens=64)
            print(label, "-> unexpectedly succeeded")
        except LLMError as e:
            print(f"{label:<14} -> {e.kind} ({e.code})")

## Part 5 — Run the automated tests for this stage

In [26]:
import subprocess, sys
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/unit/test_data_quality.py", "tests/unit/test_dq_reviewer.py"],
                     capture_output=True, text=True)
print(res.stdout[-400:] or res.stderr[-400:])
assert res.returncode == 0

..............................                                  [100%]
39 passed in 9.55s



## Troubleshooting

| Symptom | Fix |
|---|---|
| `FileNotFoundError ... Run: python scripts/generate_data.py` | Generate the data first (notebook 02 or the script). |
| `ModuleNotFoundError: google.genai` / `pydantic` | `python -m pip install -r requirements.txt` inside the venv (Python 3.11+). |
| Key "not set" although `.env` has it | Restart the kernel after editing `.env`; check the variable name is exactly `GEMINI_API_KEY`; no quotes needed. |
| Smoke test: `auth` | Key invalid or restricted. Create a new one in Google AI Studio. |
| Smoke test: `model_not_found` | Set `GEMINI_MODEL` in `.env` to a model listed in the Gemini models page. |
| `quota` / `rate_limit` | Free-tier limits reached. Wait, or enable billing in AI Studio. |
| Live outcome `escalated` with guardrail violations | Expected sometimes: the model broke a rule twice. You still get the deterministic review. Inspect `rejected_attempts`. |
| `invalid_output` with status `incomplete` | Output hit `max_output_tokens`; raise `gemini.max_output_tokens` in `configs/config.yaml`. |

## Summary and next steps

* The deterministic engine finds all 154 defective records across 20 rules with zero false positives, and separates fail / warning / error.
* The reviewer turns the report into grouped, business-readable findings; guardrails reject hallucinated rules, dropped failures, severity downgrades and unjustified "fit for use" verdicts; revisions and retries are bounded; failures escalate to a deterministic review instead of a fabricated one.

**Next:** Stage 3 loads these datasets into BigQuery; Stage 5 runs the same rule IDs as BigQuery SQL; notebook 05 adds the SQL & Analytics Reviewer and the Answer Verification Reviewer.

## References
- Gemini API structured output: https://ai.google.dev/gemini-api/docs/structured-output
- Gemini models: https://ai.google.dev/gemini-api/docs/models
- Gemini API keys: https://ai.google.dev/gemini-api/docs/api-key
- Gemini API errors: https://ai.google.dev/gemini-api/docs/api-errors
- Google Gen AI Python SDK: https://googleapis.github.io/python-genai/